# Generate Test Data
Generates data in tests/test_data/




In [1]:

#import sys
#sys.path.insert(0, "/mnt/c/Users/hans/OneDrive/Python3/packages/cdxcore") # do this before importing anything else

import os
os.environ["QUANTDATA_TEST_MODE"] = "generate_tests" # do this before importing anything else

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.sanos.arbfree
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree

from quantdata import env# triggers construction of caches.
from quantdata.biz_calendar import next_business_day
from cdxcore.verbose import Context

def generate_day(ticker, d):
    from quantdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
    from quantdata.widb.arbfree import mk_arb_free_day, MkArbFree

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    mk_arb_free = MkArbFree(timeout_seconds=60*20)
    verbose = Context.all

    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, raise_if_bad_day=False)
    enr_day = enrich_day(ticker, d, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad_day=False, override_cache_mode="off")

    buggy   = enr_day[enr_day['pure_ask_price'] <= enr_day['pure_bid_price']]
    assert buggy.empty, f"Found {len(buggy)} rows with ask price <= bid price for {ticker} {d}:\n{buggy[['expiry','strike','spot','pure_ask_price', 'pure_bid_price','call_ask','call_bid','mkt_call_ask', 'mkt_call_bid','mkt_put_ask', 'mkt_put_bid']]}"


    af_day = mk_arb_free_day(ticker, d, mk_arb_free=mk_arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad_day=False)

tickers = ["SPX", "AAPL"]
dates = ["2006-01-31", "2005-11-18", "2024-03-03", "2025-01-02", "2024-12-24", "2010-07-05", "2011-07-06"]
dates = [next_business_day(d,count_start=True) for d in dates]

for ticker in tickers:
    for d in dates:
        print(f"Generating data for {ticker} {d}")
        generate_day(ticker, d)


/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/__init__.py:292: UserWarning: Initializing quantdata. **WARNING: quantdata is in test data generation mode **.
Caching directories are set as follows:
 raw: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/raw/
 processing: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/processing/
 training: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/training/
 experiments: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/experiments/
 temp: /tmp/tmp66f1kx1b/quantdata_test_temp/
Default cache mode is set to 'on'.


Generating data for SPX 2006-01-31
00: Collecting raw market data for SPX on 2006-01-31.
Loading library list...
Done
01:   Fetching spot price for secid 108105 on 2006-01-31.
01:   Fetching spot price for secid 108105 on 2006-01-30.
01:   Fetching USD zero curve for 2006-01-31.
01:   Fetching option prices for secid 108105 on 2006-01-31.
01:   Fetching forward prices for secid 108105 on 2006-01-31.
00: SPX 2006-01-31: collected 300/641 options. Removed: 2/641 'bad' options and 19/639 options with bid <= 0.011.
00: Collecting raw market data for SPX on 2006-01-31.
01:   Fetching spot price for secid 108105 on 2006-01-25.
00: SPX 2006-01-31: collected 300/641 options. Removed: 2/641 'bad' options and 19/639 options with bid <= 0.011.
00: SPX 2006-01-31: Enriched 114 of 300 raw options across 3 of 3 input expiries.
01:   Dropped the following 7 of 10 input expiries: 2006-02-03pm (insufficient traded options), 2006-04-22am (insufficient traded strikes), 2006-09-16am (insufficient traded s